# G3 · Inferencia física

**Spec:** [`docs/spec_G3_codex_physical_inference.md`](../docs/spec_G3_codex_physical_inference.md)  |  **Bloque:** G · Caracterización  |  **Run por defecto:** `ROXs12b_realigned`

Infiere L_acc y Ṁ (multilínea, límite = más restrictivo) y ajusta plantillas (diferido).

| | |
|---|---|
| **Entrada** | G2 + relaciones de acreción + modelos |
| **Salida (QC/productos)** | `stages/stage_g3_qc.json` |
| **Consume aguas abajo** | G4, G5 |


## Qué hace G3 y qué queda diferido

G3 es la **inferencia física**: de los flujos/límites de líneas (G2) infiere la **luminosidad de acreción L_acc** y la **tasa Ṁ** (relación Hα de Alcalá 2017), y *ajustaría* plantillas/atmósferas/tracks para SpT/Teff/masa — pero eso está **diferido (pendiente de librerías externas)**.

**Acreción:** L_acc ≤ **4.1×10⁻⁶ L☉** (límite superior, de Hα — la única línea con relación en config, regla *más restrictiva*); **Ṁ p50 = 1.3×10⁻¹² M☉/yr** (MC n=2000).

**Diferencia con E3 (definicional):** G3 usa **5σ** del flujo de Hα de G2 **con** el factor de truncamiento de disco R_in=1.25; E3 usa Gumbel 99% **sin** R_in. Misma cadena física (Alcalá 2017); el desfase ~1.6× (1.3e-12 vs 8.2e-13) es de **definición**. Canónica **sin decidir** (usuario diferido, [`docs/mdot_limit_definition_note.md`](../docs/mdot_limit_definition_note.md)).

**Diferido → `not_constrained`:** atmósfera (BT-Settl), tracks (BHAC15/ATMO2020), plantillas (Luhman/Bonnefoy) — SpT/Teff/masa necesitan datos externos. **Por eso la clasificación de G4 es ambigua.** Provisional.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
python -c "from musepipe.stages.stage_g3_accretion import run_stage_g3_accretion; run_stage_g3_accretion('$RUN')"
```

Moderado (MC n=2000).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -c "from musepipe.stages.stage_g3_accretion import run_stage_g3_accretion; run_stage_g3_accretion(\'$RUN\')"'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/stage_g3_qc.json', RUN_ID)
nb.show(qc, keys=['mdot_p50_msun_yr', 'l_acc_lsun', 'combined_accretion', 'libraries', 'definition'], title='G3')


## Resultados que llevaron a la conclusión

Acreción, comparación con E3 y el estado de las librerías del `stage_g3_qc.json`.


In [ ]:
q = nb.load_qc('stages/stage_g3_qc.json', RUN_ID)
ca = q['combined_accretion']
print(f"acreción: {ca['kind']} L_acc = {ca['l_acc_lsun']:.2e} L☉ (de {ca['from_line']}, regla {ca['rule']})")
print(f"Ṁ p50 = {q['mdot_p50_msun_yr']:.2e} M☉/yr (MC n={q['mc']['n']}); {q['n_lines_with_relation']} línea con relación")
e3 = nb.load_qc('stages/stage_h03_qc.json', RUN_ID)
e3_mdot = {L['method']: L['mdot'] for L in e3['limits']}[e3['canonical_method']]
print(f"\ncomparación: E3 Ṁ={e3_mdot:.2e} (Gumbel99, sin R_in) vs G3 Ṁ={q['mdot_p50_msun_yr']:.2e} (5σ, con R_in) -> {q['mdot_p50_msun_yr']/e3_mdot:.2f}× definicional")
print('\nlibrerías (tipado espectral):')
for k, v in q['libraries'].items():
    print(f"   {k:20s} {v}")


## Plot 1 — E3 vs G3: la misma física, dos definiciones

Los dos límites de Ṁ: **E3 = 8.2×10⁻¹³** (Gumbel 99%, sin R_in) y **G3 = 1.3×10⁻¹²** (5σ, con el factor R_in 1.25). El desfase ~1.6× es puramente **definicional** — misma cadena física (Alcalá 2017). Canónica sin decidir.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_g3_qc.json', RUN_ID)
    e3 = nb.load_qc('stages/stage_h03_qc.json', RUN_ID)
    e3_mdot = {L['method']: L['mdot'] for L in e3['limits']}[e3['canonical_method']]
    g3_mdot = q['mdot_p50_msun_yr']
    fig, ax = plt.subplots(figsize=(6.5, 4.3))
    bars = ax.bar(['E3\n(Gumbel 99%,\nsin R_in)', 'G3\n(5σ,\ncon R_in 1.25)'], [e3_mdot, g3_mdot],
                  color=['tab:blue', 'tab:green'])
    for b, v in zip(bars, [e3_mdot, g3_mdot]):
        ax.text(b.get_x() + b.get_width() / 2, v * 1.02, f'{v:.2e}', ha='center', fontsize=10)
    ax.set_ylabel('Ṁ límite superior [M☉/yr]')
    ax.set_title(f'G3 · E3 vs G3: {g3_mdot/e3_mdot:.2f}× (definicional, misma física)')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g3_accretion'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'e3_vs_g3.png', dpi=110); print('figura ->', outdir / 'e3_vs_g3.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — qué constriñe G3 y qué queda diferido

G3 **computa** la acreción (L_acc, Ṁ vía Alcalá) pero **difiere** el tipado espectral (atmósfera BT-Settl, tracks, plantillas) por falta de librerías externas → SpT/Teff/masa `not_constrained` → **G4 ambigua**.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_g3_qc.json', RUN_ID)
    items = [('acreción (L_acc, Ṁ)', 'computado'),
             ('atmósfera (BT-Settl)', 'diferido'),
             ('tracks (BHAC15/ATMO2020)', 'diferido'),
             ('plantillas (Luhman/Bonnefoy)', 'diferido')]
    col = {'computado': 'tab:green', 'diferido': '0.7'}
    names = [i[0] for i in items]
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.barh(names, [1] * len(names), color=[col[i[1]] for i in items])
    for i, (n, s) in enumerate(items):
        ax.text(0.5, i, f'{n}  →  {s}', ha='center', va='center', fontsize=9, color='w' if s == 'computado' else 'k', weight='bold')
    ax.set_xlim(0, 1); ax.set_xticks([]); ax.set_yticks([]); ax.invert_yaxis()
    ax.set_title('G3 · acreción computada; tipado espectral diferido (pending_libraries)')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g3_accretion'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'characterization_status.png', dpi=110); print('figura ->', outdir / 'characterization_status.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Ṁ p50 ≈ 1.3×10⁻¹² M☉/yr** (5σ de G2 + factor R_in 1.25); difiere de E3 (8.2e-13) solo por DEFINICIÓN; canónica sin decidir. · [`docs/mdot_limit_definition_note.md`](../docs/mdot_limit_definition_note.md)
- L_acc ≤ 4.1×10⁻⁶ L☉ de Hα (única línea con relación, regla más restrictiva).
- Plantilla/atmósfera/tracks = `not_constrained` (pending_libraries: BT-Settl/BHAC15/Luhman-Bonnefoy diferidas) → G4 ambigua.


## Conclusión (registrada)

**G3: L_acc ≤ 4.1×10⁻⁶ L☉, Ṁ p50 = 1.3×10⁻¹² M☉/yr (5σ + R_in).**

- **Fecha:** 2026-07-08 (provisional).
- **vs E3:** 8.2×10⁻¹³ (Gumbel99, sin R_in) → ~1.6× por definición, no por física; canónica sin decidir.
- **Tipado espectral diferido:** atmósfera/tracks/plantillas `not_constrained` (falta de librerías externas).
- **Consecuencia:** sin SpT/Teff/masa espectroscópicos, la clasificación de G4 queda **ambigua** (planeta/BD/M no resuelto).
- **Downstream:** G4 (clasificación) y G5 (síntesis).
